# Finite-Amplitude Rossby Wave Diagnostics

**Clare S. Y. Huang** (University of Chicago), Christopher Polster (JGU Mainz),
Noboru Nakamura (University of Chicago)

This POD quantifies wave--mean-flow interaction using **finite-amplitude local
wave activity (LWA)** in the Nakamura--Huang (2018) quasi-geostrophic
formulation, via the [`falwa`](https://github.com/csyhuang/hn2016_falwa)
package.

For each season it computes, on a pseudoheight vertical coordinate
$z = -H\ln(p/p_0)$ with $H = 7000$ m and $p_0 = 1000$ hPa:

| Quantity | Meaning |
|---|---|
| `uref` | zonal-mean **reference state** zonal wind &mdash; the zonalised flow |
| `zonal_mean_u` | zonal-mean zonal wind |
| `zonal_mean_u - uref` | departure of the flow from its reference state |
| `zonal_mean_lwa` | zonal-mean local wave activity |
| `lwa_baro` | barotropic (column-integrated) LWA |
| `u_baro` | barotropic zonal wind |
| `covariance_lwa_u_baro` | covariance of `lwa_baro` and `u_baro` over time |

LWA measures Rossby wave amplitude in a finite-amplitude (not linearised)
sense; `uref` is the zonal-mean state the flow would relax to with the waves
removed. Their difference and covariance diagnose how wave activity decelerates
the jet.

---

**This notebook is the POD driver.** The framework executes it headless with
`jupyter nbconvert --execute` and renders the result to HTML. The computation
lives in `finite_amplitude_wave_diag_zonal_mean_rework.py`, imported below, so
that the notebook, the standalone script and the test harness all run the same
code.

## Setup

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

# The framework copies this notebook into WORK_DIR and executes it with
# cwd=WORK_DIR, so the POD's own modules are not importable by default.
# POD_HOME points at the POD's code directory; this is the convention used by
# diagnostics/example_notebook.
sys.path.append(os.environ["POD_HOME"])

import finite_amplitude_wave_diag_zonal_mean_rework as fawd

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

## Part 1: the framework hand-off

`load_case()` reads `case_info.yml` &mdash; whose location the framework puts in
the `case_env_file` environment variable &mdash; to find the postprocessed data
catalog and the model's own names for each variable and coordinate.

In [ ]:
ctx = fawd.load_case()

print(f"case      : {ctx.casename}")
print(f"years     : {ctx.firstyr}-{ctx.lastyr}")
print(f"variables : {ctx.u_var_name}, {ctx.v_var_name}, {ctx.t_var_name}")
print(f"coords    : {ctx.time_coord_name}, {ctx.plev_name}, {ctx.lat_name}, {ctx.lon_name}")
print(f"work dir  : {ctx.wk_dir}")
print(f"catalog   : {ctx.catalog_file}")

### What is in the data catalog?

In [ ]:
ctx.catalog.df

### The dataset the query returned

In [ ]:
ctx.model_dataset

### Vertical coordinate

The analysis grid is uniform in pseudoheight. If the input pressure levels
already form such a grid, `falwa` takes $\Delta z$ and `kmax` straight from the
data and performs no vertical interpolation of its own; otherwise the fields are
interpolated onto a uniform grid. The check below reports which path this
dataset takes.

In [ ]:
plev_hpa = ctx.model_dataset[ctx.plev_name].values
on_even_grid, dz, kmax = fawd.infer_vertical_grid(plev_hpa, default_dz=fawd.TARGET_DZ)

print(f"{len(plev_hpa)} levels, {plev_hpa.max():.2f} to {plev_hpa.min():.4f} hPa")
print(f"evenly spaced in pseudoheight : {on_even_grid}")
print(f"dz = {dz:.1f} m, kmax = {kmax}")
print("falwa vertical interpolation  :", "SKIPPED" if on_even_grid else "performed")

## Part 2: compute the diagnostics

Each season is processed one timestep at a time: read, gridfill the missing
values below ground, interpolate onto the analysis grid, run `QGField`, keep the
five diagnostics, discard the rest. Peak memory is therefore one timestep rather
than one season.

Seasons with fewer than two timesteps are skipped &mdash; partial-year input is
legitimate, and the covariance needs at least two samples.

In [ ]:
season_results = {}

for season, months in fawd.SEASON_TO_MONTHS:
    print(f"--- {season} " + "-" * 50)
    result = fawd.process_season(ctx, season, months)
    season_results[season] = result
    if not result.skipped:
        fawd.save_diagnostics(ctx, result)

computed = [s for s, r in season_results.items() if not r.skipped]
print("\nseasons computed:", computed or "none")

## Part 3: figures

Each season yields seven figures. They are written as EPS for the framework to
convert for the generated webpage, and displayed inline here.

In the lat-lon maps, grey stippling marks where values were filled by the
Poisson solver because the surface lies above the level &mdash; over high
terrain, mainly.

In [ ]:
for season, result in season_results.items():
    if result.skipped:
        print(f"{season}: skipped ({result.n_time} timestep(s))")
        continue
    print(f"=== {season}: {result.n_time} timesteps ===")
    figures = fawd.plot_season(ctx, result)
    for fig in figures.values():
        plt.show()

## Part 4: the numbers behind the figures

The seasonal means are also written to `$WORK_DIR/model/netCDF/diagnostics_<SEASON>.nc`
so they can be reanalysed without rerunning the POD.

In [ ]:
import xarray as xr

for season, result in season_results.items():
    if result.skipped:
        continue
    path = os.path.join(ctx.netcdf_dir, f"diagnostics_{season}.nc")
    with xr.open_dataset(path) as ds:
        print(f"{season}: {path}")
        for name in ds.data_vars:
            values = ds[name].values
            print(f"    {name:24s} {str(values.shape):14s} "
                  f"min {np.nanmin(values):+9.3f}  max {np.nanmax(values):+9.3f}")

## Part 5: check the run was complete

A notebook that fails mid-cell still renders as HTML up to the failure, which
can look like success. This cell makes an incomplete run explicit.

In [ ]:
skipped = [s for s, r in season_results.items() if r.skipped]
if skipped:
    print(f"WARNING: no data for {', '.join(skipped)}. "
          f"Expected if the input covers less than a full year.")

missing = []
for season, result in season_results.items():
    if result.skipped:
        continue
    for name in ("zonal_mean_u", "zonal_mean_lwa", "zonal_mean_uref",
                 "zonal_mean_delta_u", "u_baro", "lwa_baro", "u_lwa_covariance"):
        path = os.path.join(ctx.plot_dir, f"{season}_{name}.eps")
        if not os.path.isfile(path):
            missing.append(os.path.basename(path))

assert not missing, f"figures not written: {missing}"
print(f"OK: {len(computed)} season(s) computed, all figures written to {ctx.plot_dir}")

In [ ]:
ctx.model_dataset.close()
print("POD Finite-amplitude wave diagnostic (zonal mean) finished successfully!")